# Exploring a panel run: tasks, checks, borders, the panel, months against quarters, AMW

This notebook reads one run of `plan.py`, `detect.py` and (when it has run) `assemble_period.py`,
and shows

1. the run: its frozen settings, its plan, and every task's state,
2. the chip grid and the `chip_id`, checked against the tile keys,
3. every task, streamed: integrity checks and statistics per period and subregion,
4. the border tiles: scored under both subregions, so each of their chips has two rows; whether
   the two agree, and which copy the assembled panel kept,
5. the assembled panel against detect's output: the rows, the flags, the raw counts, and every
   value row by row on a sample of tiles,
6. months against their quarter: the count identity and detection agreement, on a sample of tiles,
7. detections and cloud cover through time and over the map,
8. our quarters against AMW's published detections, from `compare.py`'s results, if present,
9. what the full Amazon run would weigh.

It cross-checks `detect/` against the panel and against `compare.py`. The panel on its own,
what it holds and how to read it, is `panel_overview.ipynb`, and `check_panel.py` runs its
eight checks (README A7); this notebook does not repeat them.

It only **reads**. Half-written files start with `_` and are skipped, so it is safe while
`detect.py` or `assemble_period.py` is writing.

**Memory and time.** One period of the subregion 2 and 3 run is 46 million chips, the whole
basin 145 million, so nothing here loads a whole frequency. Sections 3 and 5 stream the part
files in batches and keep only sums and the chip IDs; sections 4, 5 and 6 read filtered slices
(the border tiles, a sample of tiles). The biggest thing held in memory is the plan's chip IDs
(about 2.5 GB for the whole basin's plan). On a laptop, the subregion 2 and 3 run takes about
20 minutes, most of it in section 3.

**Where.** On a laptop, point `RUN` at the copy under `tests_output/` (README B6 says what to
`rsync`: `plan/`, `run/`, `detect/`, and the panel folder once assembled). On Goethe the first
cell reads `$RUN` as set by the login lines, and the notebook runs in VS Code's Jupyter over
Remote-SSH (`uv pip install ipykernel` in the venv, once); its reading is light enough for the
login node, except on a full-basin run, where it belongs in a job.

In [ ]:
# ---- settings: the only cell you normally edit --------------------------------------
import os
from pathlib import Path

RUN = Path(os.environ.get("RUN", "tests_output/aca_full_2026-10-23"))   # $RUN on Goethe; else the laptop copy
SUBREGIONS = None        # e.g. [2, 3]: the subregions to look at; None = every one with a started task
FOCUS_QUARTER = None     # the quarter for sections 6 to 8, e.g. "2025Q1"; None = the first complete one
SAMPLE_TILES = 2000      # section 6 works on this many random tiles per subregion; None = all of them
EQUALITY_TILES = 200     # section 5 compares detect's values with the panel's on this many random tiles per subregion
MAX_ROWS = 60_000_000    # refuse to load more rows than this into memory at once (~3 GB)
RAW = 0.4                # AMW's raw detection: confidence strictly above this

In [ ]:
from __future__ import annotations

import csv
import inspect
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.dataset as ds
import pyarrow.parquet as pq

# The repository root: the first folder upwards that holds panel_pipeline/.
REPO = Path.cwd().resolve()
while not (REPO / "panel_pipeline").is_dir():
    if REPO.parent == REPO:
        raise SystemExit("Start Jupyter inside the AMW-mining-detector repository")
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "panel_pipeline"))          # status.py
sys.path.insert(0, str(REPO / "panel_pipeline" / "lib"))  # config, periods, read_panel, ...
import status                                              # the same reading of a task folder as status.py
from config import load_config
from periods import Period
from read_panel import read_panel
try:
    from read_panel import panel_dir                       # assemble_period.py --subregions writes a panel of part of the plan to its own folder
except ImportError:                                        # before 1 Nov 2026: one panel/ folder
    def panel_dir(run, subregions=None):
        return Path(run) / "panel"
READ_PANEL_SUBREGIONS = "subregions" in inspect.signature(read_panel).parameters

def load_panel(frequency, periods, columns, subregions):
    kw = {"subregions": subregions} if READ_PANEL_SUBREGIONS else {}
    return read_panel(RUN, frequency, periods=periods, columns=columns, **kw)

RUN = RUN if RUN.is_absolute() else (REPO / RUN).resolve()
if not (RUN / "run" / "config.yaml").is_file():
    raise SystemExit(f"{RUN} is not a planned run (no run/config.yaml)")
pd.set_option("display.width", 200, "display.max_columns", 40, "display.precision", 4)

VOTES = [f"vote_{i}" for i in range(1, 7)]
cfg = load_config(RUN / "run" / "config.yaml")                 # the frozen copy, as every script reads it
prov = json.loads((RUN / "run" / "provenance.json").read_text())
tiles = pd.read_csv(RUN / "plan" / "tiles.csv")
planned_subregions = status.subregions_with_tiles(RUN)         # every subregion of the plan, scored or not
border_keys = set(tiles["tile_key"][tiles["tile_key"].duplicated()])   # listed under two subregions

def tile_number(key: str) -> int:
    # The part of a chip_id before the last three digits (grid.py); a tile key is 552:12:10.0:zone:col:row
    zone, col, row = (int(x) for x in key.split(":")[3:6])
    return (zone * 1000 + col + 500) * 10_000 + row + 5000

tiles["tile"] = tiles["tile_key"].map(tile_number)
tiles_of = {n: set(g["tile"]) for n, g in tiles.groupby("subregion")}
print(f"{RUN}\nplanned {prov['planned_utc']} on {prov['host']}, commit {str(prov['git_commit'])[:7]}; "
      f"{tiles['tile_key'].nunique():,} tiles in subregions {planned_subregions}, {len(border_keys)} on borders between them")

## 1. The run: settings, plan and tasks

One run = one folder, made once by `plan.py`: `run/` (the frozen `config.yaml` and
`provenance.json`), `plan/` (`tiles.csv`: which tiles each subregion scores; `chips.parquet`:
every chip, its square and flags), `detect/<frequency>/period=<tag>/subregion=<n>/` (one task
each: part files, `_status.json`, `_log.txt`) and, after assembly, the panel:
`panel/<frequency>/<layer>/period=<tag>/`, or `panel_subregions_2_3/...` when
`assemble_period.py --subregions 2 3` assembled part of the plan.

A tile on the border between two subregions is in both lists and is scored twice (as AMW
does); section 4 looks at those. The task grid below is `status.py`'s, over the whole plan;
everything after it looks at the subregions with a started task (or `SUBREGIONS`).

In [ ]:
print("periods   ", {f: f"{len(p)} ({p[0].tag} to {p[-1].tag})" for f, p in cfg.periods.items()})
print("region    ", cfg.region.clip_to or "whole Amazon_ACA", "| subregions in the plan:", planned_subregions)
print("imagery   ", cfg.imagery)
print("inference ", cfg.inference)
print("thresholds", cfg.thresholds)
print("counts    ", prov["counts"])
if prov.get("git_uncommitted_files"):
    print("NOTE: planned with uncommitted files:", prov["git_uncommitted_files"])

rows = status.tasks(cfg, RUN, list(cfg.periods), planned_subregions)
tasks = pd.DataFrame([{"frequency": r["frequency"], "period": r["period"].tag, "subregion": r["subregion"],
                       "state": r["state"], "note": r["note"]} for r in rows])
for freq in cfg.periods:
    mine = [r for r in rows if r["frequency"] == freq]
    print(f"\n{freq}: " + ", ".join(f"{v} {k}" for k, v in tasks[tasks['frequency'] == freq]['state'].value_counts().items()))
    print("\n".join(status.grid(mine, freq, planned_subregions)))
print("\n" + status.LEGEND)

def task_folder(frequency: str, period: str, subregion: int) -> Path:
    return RUN / "detect" / frequency / f"period={period}" / f"subregion={subregion}"

records = []
for t in tasks.itertuples():
    path = task_folder(t.frequency, t.period, t.subregion) / "_status.json"
    if not path.is_file():
        continue
    s = json.loads(path.read_text())
    last = s["sessions"][-1]
    ended = last.get("finished_utc") or last.get("stopped_utc")
    hours = sum((pd.Timestamp(x.get("finished_utc") or x.get("stopped_utc")) - pd.Timestamp(x["started_utc"])).total_seconds()
                for x in s["sessions"] if x.get("finished_utc") or x.get("stopped_utc")) / 3600
    records.append({"frequency": t.frequency, "period": t.period, "subregion": t.subregion, "state": t.state,
                    "complete": s["complete"], "tiles_saved": s["tiles_saved"], "tiles_planned": s["tiles_planned"],
                    "tiles_failed": len(s["tiles_failed"]), "chips_saved": s["chips_saved"],
                    "sessions": len(s["sessions"]), "hours": round(hours, 2), "workers": last.get("max_workers"),
                    "limit": last.get("limit"), "ended": ended, "git_commit": str(last.get("git_commit"))[:7],
                    "image_cache": any(x.get("image_cache") for x in s["sessions"])})
started = pd.DataFrame(records, columns=["frequency", "period", "subregion", "state", "complete", "tiles_saved", "tiles_planned",
                                         "tiles_failed", "chips_saved", "sessions", "hours", "workers", "limit", "ended",
                                         "git_commit", "image_cache"])
subregions = sorted(SUBREGIONS or set(started["subregion"]))        # the subregions the rest of the notebook looks at
PANEL = panel_dir(RUN, subregions)                                  # where assemble_period.py put their panel
print(f"\nsubregions looked at: {subregions}; their panel folder: {PANEL.name}/ "
      + ("(exists)" if PANEL.is_dir() else "(nothing assembled yet)"))
started if len(started) else "no task has written a _status.json yet"

## 2. The chip grid and the chip ID

A **tile** is AMW's 552 px square (+12 px padding, 5.76 km) on a fixed global grid; the model
scores **chips** of 48 px moved 24 px at a time, 23 × 23 = 529 per tile. `chip_id` is one number
built from where the chip is, so it is the same in every period, frequency and run:

```
chip_id = zone | tile column + 500 | tile row + 5000 | chip index
          20   |      510          |     5103        |   000      ->  205105103000
```

The cell streams `plan/chips.parquet` (46 million rows for subregions 2 and 3, 145 million for
the whole basin) and takes every
`chip_id` apart again against the tile key AMW's own code wrote. If this passes, IDs are a
safe join key between periods, frequencies, runs and the assembled panel. It also keeps the
planned IDs (one sorted array) and one centre point per tile, for the sections below.

In [ ]:
def decode(ids: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    # Undo grid.chip_ids: zone | column+500 | row+5000 | chip.
    rest, chip = np.divmod(ids, 1000)
    rest, row = np.divmod(rest, 10_000)
    zone, col = np.divmod(rest, 1000)
    return zone, col - 500, row - 5000, chip

CENTRE_CHIP = 11 * 23 + 11                    # the middle chip of the 23 x 23 grid: the tile's centre
pf = pq.ParquetFile(RUN / "plan" / "chips.parquet")
n_chips = pf.metadata.num_rows
ids_parts, centres, per_tile = [], [], {}
ok = {"chip_id strictly increasing": True, "zone = tile key": True, "column = tile key": True,
      "row = tile key": True, "chip = chip_index": True}
last = -1
for batch in pf.iter_batches(batch_size=2_000_000, columns=["chip_id", "tile_key", "chip_index", "minx", "miny", "maxx", "maxy"]):
    ids = batch.column("chip_id").to_numpy()
    keys = batch.column("tile_key").to_pandas()
    zone, col, row, chip = decode(ids)
    parts = keys.str.split(":", expand=True)[[3, 4, 5]].astype(int).to_numpy()
    ok["chip_id strictly increasing"] &= bool(ids[0] > last and (np.diff(ids) > 0).all())
    ok["zone = tile key"] &= bool((zone == parts[:, 0]).all())
    ok["column = tile key"] &= bool((col == parts[:, 1]).all())
    ok["row = tile key"] &= bool((row == parts[:, 2]).all())
    ok["chip = chip_index"] &= bool((chip == batch.column("chip_index").to_numpy()).all())
    last = ids[-1]
    ids_parts.append(ids)
    for t, c in zip(*np.unique(ids // 1000, return_counts=True)):
        per_tile[t] = per_tile.get(t, 0) + int(c)
    mid = chip == CENTRE_CHIP
    centres.append(pd.DataFrame({"tile": ids[mid] // 1000,
                                 "x": (batch.column("minx").to_numpy()[mid] + batch.column("maxx").to_numpy()[mid]) / 2,
                                 "y": (batch.column("miny").to_numpy()[mid] + batch.column("maxy").to_numpy()[mid]) / 2}))
all_ids = np.concatenate(ids_parts); del ids_parts
tile_of_id = all_ids // 1000                  # the tile of every planned chip, computed once (8 bytes per chip)
centres = pd.concat(centres, ignore_index=True)
ok["529 chips per tile"] = all(c == 529 for c in per_tile.values())
ok["every planned tile has chips"] = set(per_tile) == set(tiles["tile"])
print(f"{n_chips:,} chips in {len(per_tile):,} tiles")
for name, passed in ok.items():
    print(f"   {'PASS' if passed else 'FAIL'}  {name}")

def tiles_array(n: int) -> np.ndarray:
    return np.fromiter(tiles_of[n], dtype=np.int64)

planned_ids = {n: all_ids[np.isin(tile_of_id, tiles_array(n))] for n in subregions}
pairs = [(n, m) for i, n in enumerate(subregions) for m in subregions[i + 1:] if tiles_of[n] & tiles_of[m]]
shared_ids = {(n, m): all_ids[np.isin(tile_of_id, np.fromiter(tiles_of[n] & tiles_of[m], dtype=np.int64))] for n, m in pairs}
print("planned chips per subregion:", {n: f"{len(v):,}" for n, v in planned_ids.items()},
      "| chips shared by two subregions:", {f"{n} & {m}": f"{len(v):,}" for (n, m), v in shared_ids.items()})
pf.read_row_group(0).slice(0, 3).to_pandas()

## 3. Every task, streamed: integrity checks and statistics

One row per task (frequency, period, subregion). The part files are read in batches, so a
28-million-chip task costs a few hundred MB, not gigabytes. Checks, each PASS or FAIL:

| check | meaning |
|---|---|
| `all planned chips present` | a complete task holds exactly its subregion's planned chips (unfinished tasks are missing chips by design, so not checked) |
| `no unplanned chips` | nothing outside the plan |
| `no duplicate chip` | within one task; across subregions, border chips *are* duplicated (section 4) |
| `confidence = mean(votes)` | AMW's score is the mean of the six members |
| `0 <= confidence <= 1`, `n_clear <= n_obs` | sanity |
| `no missing scores or counts` | `confidence`, the votes, `n_obs`, `n_clear` and `empty_share` are never missing or infinite |

**Masked pixels.** Tasks scored between 22 Oct and 1 Nov 2026 (all of `aca_full_2026-10-23`)
carry a `detect.py` bug: a pixel with no clear image arrived as `-inf` and was taken for a seen
pixel. In every chip with such a pixel, `cs_clear_mean` and `cirrus_mean` are `-inf`, `ndvi_mean`
and `bsi_mean` are missing, and `empty_share` stayed 0; the scores, votes, `n_obs`, `n_clear`,
`n_clear_median`, `thin_share`, `clear_day_mean` and `cs_mean` are right (README A6,
`docs/findings.md` §8). `masked_chips` is the share of such chips in each task and
`quality_missing_in` names the columns; `assemble_period.py` turns the wrong values into
missing ones, and section 5 allows for that when it compares values.

Statistics: `raw_share` is AMW's raw detection rate (`confidence > 0.4`), `loose_main` the share
at or above 0.43 (the first half of AMW's loose rule; the isolation half needs neighbours and
lives in the panel), `vote_sd` how much the six members disagree, `low_obs` the share of chips
with fewer than one clear image per pixel (there the model looks at missing data), and the
quality means as README A6 explains them. `tile_stats` (per tile) feeds the maps of section 7.

In [ ]:
QUALITY = ["n_obs", "n_clear", "n_clear_median", "empty_share", "thin_share", "clear_day_mean",
           "cs_mean", "cs_clear_mean", "ndvi_mean", "bsi_mean", "cirrus_mean"]      # detect.py's quality columns
CORE = ["confidence", *VOTES, "n_obs", "n_clear", "empty_share"]                    # never missing or infinite, whatever the run
STAT_COLUMNS = ["chip_id", *CORE, *QUALITY]

def stream_task(folder: Path) -> tuple[dict, pd.DataFrame, np.ndarray]:
    # Sums over every part file of one task, its per-tile aggregates, and its chip IDs.
    d = ds.dataset(folder, format="parquet")      # files starting with _ are skipped
    columns = [c for c in dict.fromkeys(STAT_COLUMNS) if c in d.schema.names]
    acc = {"rows": 0, "conf_sum": 0.0, "raw": 0, "loose_main": 0, "n_clear_sum": 0.0, "n_obs_sum": 0.0,
           "empty_sum": 0.0, "low_obs": 0, "vote_sd_sum": 0.0, "core_bad": 0, "masked": 0, "quality_missing_in": set(),
           "clear_gt_obs": 0, "conf_min": 1.0, "conf_max": 0.0, "mean_ok": True}
    ids, per_tile = [], []
    for batch in d.to_batches(columns=columns, batch_size=1_000_000):
        b = batch.to_pandas()
        conf = b["confidence"].to_numpy()
        core = b[[c for c in CORE if c in b]].to_numpy(dtype=np.float64)
        acc["core_bad"] += int((~np.isfinite(core)).sum())
        if "cs_clear_mean" in b:
            acc["masked"] += int(np.isneginf(b["cs_clear_mean"].to_numpy(dtype=np.float64)).sum())
        other = b[[c for c in QUALITY if c in b and c not in CORE]]
        if len(other.columns):
            acc["quality_missing_in"] |= set(other.columns[(~np.isfinite(other.to_numpy(dtype=np.float64))).any(axis=0)])
        acc["rows"] += len(b)
        acc["conf_sum"] += float(np.nansum(conf))
        acc["raw"] += int((conf > RAW).sum())
        acc["loose_main"] += int((conf >= cfg.thresholds.loose.t_main).sum())
        acc["n_clear_sum"] += float(b["n_clear"].sum()); acc["n_obs_sum"] += float(b["n_obs"].sum())
        acc["empty_sum"] += float(b["empty_share"].sum())
        acc["low_obs"] += int((b["n_clear"] < 1).sum())
        votes = b[VOTES].to_numpy(dtype=np.float64)
        acc["vote_sd_sum"] += float(votes.std(axis=1).sum())
        acc["mean_ok"] &= bool(np.allclose(conf, votes.mean(axis=1), atol=1e-6))
        acc["clear_gt_obs"] += int((b["n_clear"] > b["n_obs"] + 1e-4).sum())
        acc["conf_min"] = min(acc["conf_min"], float(np.nanmin(conf))); acc["conf_max"] = max(acc["conf_max"], float(np.nanmax(conf)))
        ids.append(b["chip_id"].to_numpy())
        per_tile.append(b.assign(tile=b["chip_id"] // 1000, raw=conf > RAW)
                         .groupby("tile").agg(conf_max=("confidence", "max"), raw=("raw", "sum"),
                                              n_clear=("n_clear", "sum"), chips=("chip_id", "size")))
    ids = np.sort(np.concatenate(ids)) if ids else np.array([], dtype=np.int64)   # part files are not sorted
    per_tile = (pd.concat(per_tile).groupby(level=0).agg(conf_max=("conf_max", "max"), raw=("raw", "sum"),
                                                          n_clear=("n_clear", "sum"), chips=("chips", "sum"))
                if per_tile else pd.DataFrame(columns=["conf_max", "raw", "n_clear", "chips"]))
    per_tile["n_clear"] = per_tile["n_clear"] / per_tile["chips"]
    return acc, per_tile, ids

stats_rows, tile_stats = [], []
for t in started[started["subregion"].isin(subregions)].itertuples():
    folder = task_folder(t.frequency, t.period, t.subregion)
    if not list(folder.glob("part_*.parquet")):
        continue
    acc, per_tile, ids = stream_task(folder)
    want = planned_ids[t.subregion]
    n = acc["rows"]
    stats_rows.append({
        "frequency": t.frequency, "period": t.period, "subregion": t.subregion, "complete": t.complete,
        "rows": n, "chips_saved": t.chips_saved, "rows = chips_saved": n == t.chips_saved,
        "all planned chips present": bool(np.array_equal(ids, want)) if t.complete else None,
        "no unplanned chips": bool(np.isin(ids, want, assume_unique=False).all()),
        "no duplicate chip": bool((np.diff(ids) > 0).all()),
        "confidence = mean(votes)": acc["mean_ok"],
        "0 <= confidence <= 1": 0 <= acc["conf_min"] and acc["conf_max"] <= 1,
        "n_clear <= n_obs": acc["clear_gt_obs"] == 0,
        "no missing scores or counts": acc["core_bad"] == 0,
        "conf_mean": acc["conf_sum"] / n, "raw_count": acc["raw"], "raw_share": acc["raw"] / n,
        "loose_main": acc["loose_main"] / n, "vote_sd": acc["vote_sd_sum"] / n,
        "n_obs": acc["n_obs_sum"] / n, "n_clear": acc["n_clear_sum"] / n,
        "empty_share": acc["empty_sum"] / n, "low_obs": acc["low_obs"] / n,
        "masked_chips": acc["masked"] / n, "quality_missing_in": ", ".join(sorted(acc["quality_missing_in"])),
    })
    tile_stats.append(per_tile.reset_index().assign(frequency=t.frequency, period=t.period, subregion=t.subregion))
    del ids
stats = pd.DataFrame(stats_rows)
tile_stats = pd.concat(tile_stats, ignore_index=True) if tile_stats else pd.DataFrame()
checks = [c for c in stats.columns if c.startswith(("all ", "no ", "rows =", "confidence =", "0 <=", "n_clear <="))]
if len(stats):
    failed = stats[checks].eq(False).any(axis=1).sum()
    print("ALL CHECKS PASS" if not failed else f"{failed} TASK(S) FAIL A CHECK: look for False below")
    display(stats[["frequency", "period", "subregion", "complete", "rows", *checks]])
    stats["period_start"] = stats["period"].map(lambda t: Period.parse(t).start)
    display(stats[["frequency", "period", "subregion", "rows", "conf_mean", "raw_count", "raw_share", "loose_main",
                   "vote_sd", "n_obs", "n_clear", "empty_share", "low_obs", "masked_chips", "quality_missing_in"]]
            .style.format({"rows": "{:,}", "raw_count": "{:,}", "raw_share": "{:.3%}", "loose_main": "{:.3%}",
                           "empty_share": "{:.2%}", "low_obs": "{:.2%}", "masked_chips": "{:.1%}"}))
else:
    print("no task has part files yet")

**The columns detect wrote.** A few real rows of one complete task (the first one listed above),
every column, and what each column means. The quality columns from `n_obs` to `cirrus_mean`
exist in runs scored since 23 Oct 2026; an older run has fewer. The same descriptions, for the
assembled panel, are in `panel_overview.ipynb` section 2 (`panel_pipeline/lib/variables.py`).

In [ ]:
from variables import dictionary

done = started[started["complete"]].sort_values(["frequency", "period", "subregion"]) if len(started) else started
if not len(done):
    print("no complete task yet")
else:
    t = done.iloc[0]
    first_part = sorted(task_folder(t["frequency"], t["period"], t["subregion"]).glob("part_*.parquet"))[0]
    sample = pq.ParquetFile(first_part).read_row_group(0).to_pandas().head(5)
    print(f"{t['frequency']} {t['period']}, subregion {t['subregion']}: {first_part.name}, first 5 rows "
          f"of {len(sample.columns)} columns")
    with pd.option_context("display.max_columns", None, "display.width", 250):
        display(sample)
    display(dictionary(list(sample.columns)).style.hide(axis="index")
            .set_properties(**{"text-align": "left", "white-space": "pre-wrap", "vertical-align": "top"}))

## 4. Border tiles: scored twice, once in the panel

A tile on the border between two subregions is listed under both, so `detect.py` scores it twice,
in two tasks, from the same Earth Engine request recipe. The two copies should be identical, or
nearly: Earth Engine's data for recent months can change between the two scoring dates. The
assembled panel keeps one row per chip, by AMW's `dedupe` rule (`postprocess.py:146`): the copy
with the higher confidence, and on a tie the lower subregion; its `subregion` column says which.
The cell measures the copies in `detect/`, then checks the panel's choice against them and shows
what the panel's own `_record.json` counted. A difference between copies is not an error in the
pipeline, but it is the first thing to know before reading a change near a border.

In [ ]:
border_tiles = np.fromiter((tile_number(k) for k in border_keys), dtype=np.int64)
print(f"{len(border_tiles):,} border tiles in the plan; between the subregions looked at: "
      + ", ".join(f"{n} & {m}: {len(v):,} chips" for (n, m), v in shared_ids.items()))
border_rows, dropped_raw = [], {}       # dropped_raw: per period, detections among the border copies the panel drops
for (freq, period), g in started[started["complete"] & started["subregion"].isin(subregions)].groupby(["frequency", "period"]):
    done = set(g["subregion"])
    for n, m in pairs:
        if not {n, m} <= done:
            continue
        shared = shared_ids[(n, m)]
        copies = [ds.dataset(task_folder(freq, period, sub), format="parquet")
                    .to_table(columns=["chip_id", "confidence", "n_obs", "n_clear"], filter=ds.field("chip_id").isin(pa.array(shared)))
                    .to_pandas() for sub in (n, m)]
        both = copies[0].merge(copies[1], on="chip_id", suffixes=(f"_{n}", f"_{m}"), validate="one_to_one")
        cn, cm = both[f"confidence_{n}"], both[f"confidence_{m}"]
        gap = (cn - cm).abs()
        dropped_raw[(freq, period)] = dropped_raw.get((freq, period), 0) + int((np.minimum(cn, cm) > RAW).sum())
        row = {"frequency": freq, "period": period, "subregions": f"{n} & {m}", "chips": len(both),
               "both copies present": len(both) == len(shared),
               "identical scores": float((gap == 0).mean()), "max |score gap|": float(gap.max()),
               "flips across 0.4": int(((cn > RAW) != (cm > RAW)).sum()),
               "identical n_obs": float((both[f"n_obs_{n}"] == both[f"n_obs_{m}"]).mean())}
        layer = PANEL / freq / "scores" / f"period={period}"
        if (layer / "_record.json").is_file():
            try:
                rec = json.loads((layer / "_record.json").read_text()).get("border", {})
                kept = pd.read_parquet(layer / "data.parquet", columns=["chip_id", "subregion", "confidence"],
                                       filters=[("chip_id", "in", shared)])
                k = both.merge(kept, on="chip_id", validate="one_to_one")
                hi = np.maximum(k[f"confidence_{n}"], k[f"confidence_{m}"])
                tie = k[f"confidence_{n}"] == k[f"confidence_{m}"]
                row |= {"panel kept the higher": bool((k["confidence"] == hi).all()),
                        "ties from the lower subregion": bool((k.loc[tie, "subregion"] == min(n, m)).all()),
                        "panel kept from": dict(k["subregion"].value_counts().sort_index()),
                        "record: copies differ (any column)": rec.get("copies_differ"), "record: largest gap": rec.get("largest_confidence_gap")}
            except Exception as e:                        # a panel layout this notebook does not know yet
                row["panel"] = f"could not read: {str(e)[:100]}"
        border_rows.append(row)
border = pd.DataFrame(border_rows)
border if len(border) else "no period is complete for two subregions that share tiles"

## 5. The assembled panel against detect's output

`assemble_period.py` writes, per period, two layers with the same rows in the same order, one
per planned chip of the assembled subregions, sorted by `chip_id`: `scores` (detect's values,
`subregion`, and `scored`: False for chips of tiles that failed every try) and `postprocess`
(AMW's flags `det_raw`, `det_andes`, `det_loose`, `det_stringent`, with the neighbour distances
they rest on; missing where not scored). `check_panel.py` checks the panel's own consistency;
this cell checks it against `detect/`, per period:

- the rows are exactly the plan's chips of these subregions, once each, in order;
- `det_raw` is `confidence > 0.4` wherever `scored`, and the flags are missing exactly where not;
- the panel's raw detections equal detect's once the border copies are resolved (the sum over
  subregions, minus the detections among the dropped copies, from section 4);
- **values row by row**, on `EQUALITY_TILES` random tiles per subregion: every column of the
  `scores` layer equals detect's kept copy exactly, except the five columns the assembly corrects
  for the masked-pixel bug of section 3 (`empty_share`, `cs_clear_mean`, `ndvi_mean`, `bsi_mean`,
  `cirrus_mean`), which are compared after applying the same rule (`mark_lost_quality` in
  `assemble_period.py`: `empty_share` missing where `cs_clear_mean` was `-inf` and 1 where
  `n_clear` is 0; the four others missing where not finite).

Not a difference: 368 chip pairs at the 60°W zone seam have two `chip_id`s but one AMW centre
key. Both rows stay in the panel; the lower-scored one gets no loose or stringent flag
(`twins_dropped` in the postprocess record). `det_raw` is untouched by that.

In [ ]:
def expected_raw(freq: str, period: str) -> int | None:
    # detect's raw detections for the period with border chips counted once (the kept copy)
    mine = stats[(stats["frequency"] == freq) & (stats["period"] == period)] if len(stats) else pd.DataFrame()
    if not len(mine) or not mine["complete"].all() or set(mine["subregion"]) != set(subregions):
        return None
    return int(mine["raw_count"].sum()) - dropped_raw.get((freq, period), 0)

MASKED = ["empty_share", "cs_clear_mean", "ndvi_mean", "bsi_mean", "cirrus_mean"]   # assemble_period.mark_lost_quality changes these
EXACT = ["confidence", *VOTES, "n_obs", "n_clear", "n_clear_median", "thin_share", "clear_day_mean", "cs_mean"]

def lost_quality(d: pd.DataFrame) -> pd.DataFrame:
    # The assembly's rule for chips scored with the masked-pixel bug: the wrong values become missing.
    d = d.copy()
    d.loc[np.isneginf(d["cs_clear_mean"].to_numpy(dtype=np.float64)), "empty_share"] = np.nan
    d.loc[d["n_clear"] == 0, "empty_share"] = 1.0
    for c in MASKED[1:]:
        v = d[c].to_numpy(dtype=np.float64)
        d[c] = np.where(np.isfinite(v), v, np.nan)
    return d

rng = np.random.default_rng(1)
sample_tiles = np.concatenate([rng.choice(tiles_array(n), min(EQUALITY_TILES, len(tiles_of[n])), replace=False) for n in subregions]) if subregions else np.array([], dtype=np.int64)
equality_ids = all_ids[np.isin(tile_of_id, sample_tiles)]

def detect_rows(freq: str, period: str) -> pd.DataFrame:
    # detect's rows for the sample tiles; a border chip's two copies resolved as the assembly does
    parts = []
    for sub in subregions:
        d = ds.dataset(task_folder(freq, period, sub), format="parquet")
        cols = [c for c in ["chip_id", *EXACT, *MASKED] if c in d.schema.names]
        parts.append(d.to_table(columns=cols, filter=ds.field("chip_id").isin(pa.array(equality_ids))).to_pandas().assign(subregion=sub))
    d = pd.concat(parts).sort_values(["chip_id", "confidence", "subregion"], ascending=[True, False, True])
    return d.drop_duplicates("chip_id").set_index("chip_id")

panel_rows = []
for freq in cfg.periods:
    base = PANEL / freq / "scores"
    for folder in sorted(base.glob("period=*")) if base.is_dir() else []:
        if not (folder / "_record.json").is_file():
            continue
        period = folder.name.removeprefix("period=")
        try:
            p = load_panel(freq, [period], {"scores": ["subregion", "scored", "confidence"],
                                            "postprocess": ["det_raw", "det_andes", "det_loose", "det_stringent"]}, subregions)
        except Exception as e:                            # a panel layout this notebook does not know yet
            panel_rows.append({"frequency": freq, "period": period, "error": f"could not read: {str(e)[:120]}"})
            continue
        scored = p["scored"].to_numpy()
        raw_from_conf = (p["confidence"].to_numpy() > RAW) & scored
        exp = expected_raw(freq, period)
        planned_here = np.unique(np.concatenate([planned_ids[n] for n in subregions])) if subregions else all_ids[:0]
        row = {
            "frequency": freq, "period": period, "rows": len(p),
            "rows = planned chips": len(p) == len(planned_here),
            "chip_id = plan, in order": bool(np.array_equal(p["chip_id"].to_numpy(), planned_here)),
            "scored": float(scored.mean()),
            "det_raw = confidence > 0.4": bool((p["det_raw"].fillna(False).to_numpy(dtype=bool) == raw_from_conf).all()),
            "flags missing exactly where unscored": bool((p["det_raw"].isna().to_numpy() == ~scored).all()),
            "det_raw": int(p["det_raw"].sum()), "detect's raw, borders resolved": exp,
            "panel = detect": (int(p["det_raw"].sum()) == exp) if exp is not None else None,
            "det_loose": int(p["det_loose"].sum()), "det_stringent": int(p["det_stringent"].sum()),
            "det_andes": int(p["det_andes"].sum()),
            "subregions": dict(p["subregion"].value_counts().sort_index()),
        }
        del p
        if len(equality_ids):                             # values row by row, on the sample tiles
            try:
                theirs = pd.read_parquet(folder / "data.parquet", filters=[("chip_id", "in", equality_ids)]).set_index("chip_id")
                theirs = theirs[theirs["scored"].to_numpy(dtype=bool)]
                mine = detect_rows(freq, period).reindex(theirs.index)
                differ = [c for c in EXACT if c in mine and c in theirs
                          and not np.array_equal(mine[c].to_numpy(dtype=np.float64), theirs[c].to_numpy(dtype=np.float64), equal_nan=True)]
                masked = [c for c in MASKED if c in mine and c in theirs]
                if masked:
                    fixed = lost_quality(mine)
                    differ += [f"{c} (after the rule)" for c in masked
                               if not np.allclose(fixed[c].to_numpy(dtype=np.float64), theirs[c].to_numpy(dtype=np.float64), atol=1e-6, equal_nan=True)]
                if "subregion" in theirs and not np.array_equal(mine["subregion"].to_numpy(), theirs["subregion"].to_numpy()):
                    differ.append("subregion (the kept copy)")
                row |= {"sample chips": len(theirs), "values = detect": not differ, "differ in": ", ".join(differ)}
            except Exception as e:
                row |= {"values = detect": None, "differ in": f"could not compare: {str(e)[:100]}"}
        panel_rows.append(row)
panel = pd.DataFrame(panel_rows)
if len(panel):
    flags = [c for c in panel.columns if panel[c].dtype == bool or c in ("panel = detect", "values = detect")]
    print("ALL PANEL CHECKS PASS" if not panel[flags].eq(False).any().any() else "SOME PANEL CHECKS FAIL: see the False cells")
    display(panel)
else:
    print(f"nothing assembled yet under {PANEL} (assemble_period.py, README A7)")

## 6. Months against their quarter, on a sample of tiles

Both frequencies come from the same run, so they share `chip_id`. A quarter's window is exactly
its three months' windows put together, so per pixel the quarter's image counts equal the sum of
its months' counts; means over the same pixels add up the same way, so the identity holds per chip
(`findings.md` §1). The scores do not add up: a quarter's median picture is its own composite.
A mismatch in the counts would mean a date bug, or Earth Engine data that changed between the
two scoring dates.

`SAMPLE_TILES` random tiles per subregion keep this light (2,000 tiles = 1.06 million chips per
period); `None` takes every tile.

In [ ]:
quarters_done = [p for p in started[(started["frequency"] == "quarterly") & started["complete"]]["period"].unique()]
focus = FOCUS_QUARTER or (quarters_done[0] if quarters_done else None)
subs = subregions
full = pd.DataFrame()
if focus is None:
    print("no complete quarter yet")
else:
    months = [m.tag for m in Period.parse(focus).months()]
    frames = []
    for sub in subs:
        have = {p for p in [focus, *months]
                if (task_folder("quarterly" if p == focus else "monthly", p, sub) / "_status.json").is_file()
                and json.loads((task_folder("quarterly" if p == focus else "monthly", p, sub) / "_status.json").read_text())["complete"]}
        if focus not in have or not (set(months) & have):
            print(f"subregion {sub}: {focus} or its months not complete, skipped")
            continue
        tile_pool = np.fromiter(tiles_of[sub], dtype=np.int64)
        if SAMPLE_TILES and SAMPLE_TILES < len(tile_pool):
            tile_pool = np.random.default_rng(0).choice(tile_pool, SAMPLE_TILES, replace=False)
        sample_ids = pa.array(all_ids[np.isin(all_ids // 1000, tile_pool)])
        def load(freq, period):
            d = ds.dataset(task_folder(freq, period, sub), format="parquet")
            t = d.to_table(columns=["chip_id", "confidence", "n_obs", "n_clear", "empty_share"],
                           filter=ds.field("chip_id").isin(sample_ids))
            if t.num_rows > MAX_ROWS:
                raise MemoryError(f"{t.num_rows:,} rows > MAX_ROWS; lower SAMPLE_TILES")
            return t.to_pandas().replace([np.inf, -np.inf], np.nan)     # masked pixels were once written as -inf
        q = load("quarterly", focus).rename(columns={"confidence": "conf_q", "n_obs": "n_obs_q", "n_clear": "n_clear_q", "empty_share": "empty_q"})
        m = pd.concat([load("monthly", p).assign(month=p) for p in months if p in have])
        agg = (m.groupby("chip_id").agg(months=("month", "nunique"), n_obs_m=("n_obs", "sum"), n_clear_m=("n_clear", "sum"),
                                        n_obs_ok=("n_obs", "count"), n_clear_ok=("n_clear", "count"),   # months with a finite count
                                        conf_max_m=("confidence", "max"), conf_mean_m=("confidence", "mean"))
                .reset_index())
        frames.append(agg.merge(q, on="chip_id", validate="one_to_one").assign(subregion=sub))
        print(f"subregion {sub}: {len(tile_pool):,} tiles, {len(q):,} chips, months present: {sorted(set(months) & have)}")
    if frames:
        full = pd.concat(frames, ignore_index=True)
        three = full[full["months"] == len(months)]
        for c in ("n_obs", "n_clear"):
            ok = three[f"{c}_ok"] == len(months)                            # a finite count in every month and the quarter
            diff = (three[f"{c}_m"] - three[f"{c}_q"]).abs()[ok].dropna()
            print(f"{c}: sum of the {len(months)} months vs the quarter over {len(diff):,} chips: "
                  f"max |diff| = {diff.max():.4f}, equal within 0.001: {(diff <= 1e-3).mean():.3%} ({int((diff > 1e-3).sum()):,} chips differ)")
        print(f"\n{focus}: detected in the quarter (rows) vs in any of its months (columns):")
        display(pd.crosstab(three["conf_q"] > RAW, three["conf_max_m"] > RAW, rownames=["quarter > 0.4"], colnames=["any month > 0.4"]))
        fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
        ax[0].scatter(three["conf_mean_m"], three["conf_q"], s=2, alpha=0.3)
        ax[0].plot([0, 1], [0, 1], lw=0.8, color="grey")
        ax[0].set(xlabel="mean of the monthly scores", ylabel="quarterly score", title="Same chip: months vs quarter")
        ax[1].scatter(three["n_clear_m"], three["n_clear_q"], s=2, alpha=0.3)
        lim = max(three["n_clear_q"].max(), three["n_clear_m"].max()) + 0.5
        ax[1].plot([0, lim], [0, lim], lw=0.8, color="grey")
        ax[1].set(xlabel="sum of the monthly n_clear", ylabel="quarterly n_clear", title="Count identity (on the line)")
        plt.tight_layout()

## 7. Through time and over the map

Per subregion and period, from section 3: the raw detection share, clear images per pixel and
the share of pixels the model never saw. A cloudy month shows as low `n_clear` and high
`empty_share`, and its detections are the model reading a half-black picture: compare before
reading a jump as mining. The maps are per tile (the strongest chip, or the number of raw
detections), for the focus quarter and its months.

In [ ]:
if len(stats):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for (freq, sub), g in stats.sort_values("period_start").groupby(["frequency", "subregion"]):
        style = "o-" if freq == "monthly" else "s--"
        axes[0].plot(g["period_start"], g["raw_share"], style, label=f"{freq} s{sub}")
        axes[1].plot(g["period_start"], g["n_clear"], style, label=f"{freq} s{sub}")
        axes[2].plot(g["period_start"], g["empty_share"], style, label=f"{freq} s{sub}")
    axes[0].set(title="share of chips > 0.4 (raw detection)", ylabel="share")
    axes[1].set(title="clear images per pixel (mean n_clear)")
    axes[2].set(title="share of pixels never seen clear (empty_share)")
    for a in axes:
        a.legend(fontsize=8); a.tick_params(axis="x", rotation=45); a.grid(alpha=0.3)
    plt.tight_layout()

In [ ]:
if focus is not None and len(tile_stats):
    views = [("quarterly", focus)] + [("monthly", m.tag) for m in Period.parse(focus).months()]
    fig, axes = plt.subplots(2, 2, figsize=(13, 11), sharex=True, sharey=True)
    for ax, (freq, per) in zip(axes.flat, views):
        sel = tile_stats[(tile_stats["frequency"] == freq) & (tile_stats["period"] == per)]
        sel = sel.sort_values("conf_max").drop_duplicates("tile", keep="last").merge(centres, on="tile")   # border tiles: the higher copy
        if sel.empty:
            ax.set_title(f"{freq} {per}: not scored"); continue
        sc = ax.scatter(sel["x"], sel["y"], c=sel["conf_max"], s=3, cmap="magma", vmin=0, vmax=1)
        ax.set(title=f"{freq} {per}: {int(sel['raw'].sum()):,} chips > 0.4 in {int((sel['raw'] > 0).sum()):,} tiles", aspect="equal")
    fig.colorbar(sc, ax=axes, label="strongest chip of the tile (confidence)", shrink=0.6)
    sel = tile_stats[(tile_stats["frequency"] == "quarterly") & (tile_stats["period"] == focus)].drop_duplicates("tile").merge(centres, on="tile")
    fig, ax = plt.subplots(figsize=(7, 6))
    sc = ax.scatter(sel["x"], sel["y"], c=sel["n_clear"], s=3, cmap="viridis")
    ax.set(title=f"{focus}: clear images per pixel, by tile", aspect="equal")
    fig.colorbar(sc, ax=ax, label="mean n_clear")

## 8. Against AMW's published detections

`amw_comparison/compare.py` joins our quarters with AMW's raw detections (confidence > 0.4) on
AMW's own chip key and writes its results into `<run>/amw_comparison/`: `joins.csv` (the join's
counts per quarter and subregion), `amw_files.csv` (which AMW files, with fingerprints) and
`chips_<quarter>.parquet` (every chip that we, AMW, or one of our months detected, one row per
location, border copies resolved). The notebook `amw_comparison/compare.ipynb` goes deeper; here
the counts are shown next to this run's, and cross-checked: the chips we detect in that file must
be the panel's raw detections of section 5 (or detect's with borders resolved).

In [ ]:
AMW_DIR = RUN / "amw_comparison"
if not (AMW_DIR / "joins.csv").is_file():
    print(f"no compare.py results under {AMW_DIR} (amw_comparison/README.md says how to make them)")
elif focus is None or not (AMW_DIR / f"chips_{focus}.parquet").is_file():
    print(f"compare.py results exist, but not for {focus}: {sorted(p.name for p in AMW_DIR.glob('chips_*.parquet'))}")
else:
    print("AMW files:"); display(pd.read_csv(AMW_DIR / "amw_files.csv"))
    print("the join, per quarter and subregion:"); display(pd.read_csv(AMW_DIR / "joins.csv"))
    c = pd.read_parquet(AMW_DIR / f"chips_{focus}.parquet")
    ours, theirs = c["confidence"] > RAW, c["amw_confidence"] > RAW
    c["outcome"] = np.select([ours & theirs, ours & ~theirs, ~ours & theirs], ["both", "only ours", "only AMW"], "neither (a month only)")
    table = c.groupby(["subregion", "outcome"]).size().unstack(fill_value=0)
    table["ours"] = table.get("both", 0) + table.get("only ours", 0)
    table["AMW"] = table.get("both", 0) + table.get("only AMW", 0)
    display(table)
    ours_total = int(ours.sum())
    ref = panel[(panel["frequency"] == "quarterly") & (panel["period"] == focus)] if len(panel) else pd.DataFrame()
    expected = int(ref["det_raw"].iloc[0]) if len(ref) else expected_raw("quarterly", focus)
    print(f"{focus}: compare.py counts {ours_total:,} chips of ours > 0.4; "
          + (f"{'the panel' if len(ref) else 'detect, borders resolved,'} has {expected:,}: "
             + ("MATCH" if expected == ours_total else "MISMATCH (compare.py has one row per AMW key, so the 60°W seam twins count once there; see joins.csv)")
             if expected is not None else "no complete reference to check against"))
    if "months_detected" in c:
        print(f"\nof the chips AMW detects in {focus}, how many of our months detect them too:")
        display(pd.crosstab(theirs, c["months_detected"], rownames=["AMW > 0.4"], colnames=["months > 0.4"]))

## 9. What the full Amazon run would weigh

Bytes per row measured on this run's part files, scaled to Amazon_ACA (about 145 million chips,
105 months + 35 quarters). Small files compress a little worse than big ones, so this errs high.

In [ ]:
parts = list((RUN / "detect").rglob("part_*.parquet"))
rows = sum(pq.ParquetFile(p).metadata.num_rows for p in parts)
size = sum(p.stat().st_size for p in parts)
if rows:
    bpr = size / rows
    CHIPS_ACA, MONTHS, QUARTERS = 145e6, 105, 35
    print(f"measured: {rows:,} rows in {len(parts)} part files, {bpr:.1f} bytes per row on disk (zstd), {size / 1e9:.2f} GB")
    for name, n in (("monthly", MONTHS), ("quarterly", QUARTERS)):
        print(f"{name:<10} {CHIPS_ACA * n / 1e9:5.1f} billion rows  ~{CHIPS_ACA * n * bpr / 1e9:6.0f} GB on disk")
layers = list(PANEL.rglob("data.parquet")) if PANEL.is_dir() else []
if layers:
    print(f"panel so far ({PANEL.name}/): {len(layers)} layer files, {sum(p.stat().st_size for p in layers) / 1e9:.2f} GB")

**Reading the full panel later:** never `pd.read_parquet` the whole `detect/` folder. Filter
while reading, so only the needed files and columns leave the disk:

```python
d = ds.dataset(RUN / "detect" / "monthly", format="parquet", partitioning="hive")
t = d.to_table(columns=["chip_id", "confidence", "n_clear"],
               filter=(ds.field("period") == "2025-01") & (ds.field("subregion") == 3))
```

The assembled panel reads layer by layer with `read_panel(RUN, "monthly", periods=[...], columns={...})`
(`panel_pipeline/lib/read_panel.py`), and the same works in DuckDB and in R with `arrow::open_dataset()`.